In [1]:
# Load the dataset into a DataFrame so we can explore it.
import pandas as pd
import os

csv_path = '/Users/mahamkhawar/hcde530/week-05/app_reviews_demo.csv'
df = pd.read_csv(csv_path)

The dataset is loaded. We have 500 rows and 10 columns covering app reviews across multiple UX research tools.

## Question 1: What does your dataset look like? `head()`, `info()`

In [2]:
# Question 1: What kind of data is in each row?
# I want to see if the review text, rating, and app info are consistent and complete before doing any analysis.
df.head()

,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3


Each row is one review. You can see the app name, category, star rating, written review text, date, and whether the purchase was verified. The `device_type` and `app_version` columns sometimes appear blank — that's a sign of missing data we'll check later.

In [3]:
# Question 1 (continued): How many rows do we have, and are all the columns usable?
# If a column shows fewer than 500 non-null entries, it has gaps that could affect my analysis.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


`info()` confirms we have 500 reviews. Most columns are fully filled in. Notice `device_type` and `app_version` show fewer than 500 non-null entries — those two columns have gaps.

## Question 2: What is the distribution of your most important column?

The most important column is `rating` — the core measure of user satisfaction.

In [4]:
# Question 2: Do users generally rate these apps positively or negatively?
# value_counts() tells me how reviews are distributed across 1-5 stars.
# If most ratings are 4-5, users are largely satisfied. If 1-2 dominate, there are real widespread problems.
print(df['rating'].value_counts().sort_index())
print()
print(df['rating'].describe())

rating
1     29
2     43
3     61
4    160
5    207
Name: count, dtype: int64

count    500.000000
mean       3.946000
std        1.184013
min        1.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64


Ratings skew positive: most reviews are 4 or 5 stars. 1-star reviews are rare. This is common in app stores where satisfied users are more likely to leave reviews than neutral ones.

## Question 3: Filter a meaningful subset. What's in it?

Low-rating reviews (1-2 stars) from verified purchasers — the most credible negative signals.

In [6]:
# Question 3: What do the most credible complaints look like?
# I'm filtering to verified buyers who gave 1-2 stars — these are real users with genuine negative experiences,
# not anonymous one-off reviews. Reading these helps identify patterns in what frustrates users most.
low_verified = df[(df['rating'] <= 2) & (df['verified_purchase'] == True)]
print(f'Rows in subset: {len(low_verified)}')
print()
low_verified[['app', 'category', 'rating', 'review', 'helpful_votes']].head(10)

Rows in subset: 55



,app,category,rating,review,helpful_votes
0,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,37
1,Fieldkit,field research,2,Search results are slow when the repository is...,12
26,Fieldkit,field research,2,No built-in way to generate a structured debri...,8
31,Maze,usability testing,1,Session sharing links occasionally expire befo...,10
32,Lookback,user research,2,Loading large projects takes noticeably longer...,15
42,Fieldkit,field research,2,Search results are slow when the repository is...,9
51,Maze,usability testing,1,Session sharing links occasionally expire befo...,32
74,Miro,collaborative whiteboard,1,I've lost tags twice after a session due to a ...,14
79,Dovetail,research repository,2,The Figma integration is read-only; I can't pu...,2
102,Miro,collaborative whiteboard,2,Storage limits hit quickly when you're recordi...,16


This subset contains only reviews where users gave 1 or 2 stars AND confirmed they actually used the product. These are the most reliable complaints — worth reading carefully for UX issues.

## Question 4: Group by category and find the average of a numeric column.

In [7]:
# Question 4: Which app categories have the highest and lowest user satisfaction?
# groupby lets me compare average ratings across categories — a lower score suggests that category
# has more usability problems or unmet expectations worth investigating.
avg_by_category = df.groupby('category')['rating'].mean().sort_values(ascending=False).round(2)
print('Average rating by category:')
print(avg_by_category)

Average rating by category:
category
research repository         4.12
collaborative whiteboard    4.02
usability testing           4.00
user research               3.90
field research              3.67
Name: rating, dtype: float64


Each app category has a slightly different average rating. Higher-rated categories may have more polished tools or a more satisfied user base. Lower-rated ones may signal unmet expectations or usability problems.

## Question 5: Where are the missing values? Are any columns incomplete?

In [5]:
# Question 5: Which columns can I actually rely on for analysis?
# isnull().sum() shows me where data is missing. If device_type is missing for many rows,
# I can't confidently compare mobile vs desktop experience — I'd need to note that as a limitation.
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(1)
summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(summary[summary['missing_count'] > 0])

             missing_count  missing_pct
device_type             63         12.6
app_version            111         22.2


`device_type` is missing for 63 rows (12.6%) and `app_version` for 111 rows (22.2%). All other columns are complete. Missing device info could limit any analysis comparing mobile vs. desktop experience.